# Lab Week 3 · Shape and Density
Run cells in order with **Shift + Enter**. Read the comments and add your reasoning in the explanation cells. The data are synthetic. No known group labels are passed to the clustering algorithms. Save your editable notebook and use the lesson’s PDF button for a submission copy. Nothing is submitted from this website.

In [ ]:
# Load scientific libraries into the browser's Python environment.
# First startup needs internet access; wait for this cell to finish.
import piplite
await piplite.install(["numpy", "pandas", "matplotlib", "scikit-learn", "scipy"])
print("Libraries ready. Continue with Task 1.")

## Task 1 · Inspect and prepare the observations
We create curved groups and a few scattered observations. The second feature is recorded at a different numerical scale. Inspect and standardise before comparing Euclidean distances. Labels returned by the synthetic-data generator are deliberately discarded.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, BisectingKMeans
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Fix seeds so repeated runs use the same observations.
rng = np.random.default_rng(31)
coordinates, _ = make_moons(n_samples=160, noise=0.055, random_state=31)
# Add scattered observations; their generating process does not prove they are errors.
coordinates = np.vstack([coordinates, rng.uniform([-1.4, -0.8], [2.4, 1.4], (12, 2))])
raw = coordinates * np.array([1.0, 15.0])
df = pd.DataFrame(raw, columns=["feature_1", "feature_2"])
print("Original measurements (first ten rows)")
print(df.head(10))
print("\nData types")
print(df.dtypes)
print("\nDisplay Missing values")
print(df.isna().sum())
print("\nFeature summary")
print(df.describe().round(3))

# Fit one scaler. Reuse X for every guided clustering comparison below.
scaler = StandardScaler()
X = scaler.fit_transform(df)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, values, title in zip(axes, [raw, X], ["Before scaling", "After standardisation"]):
    ax.scatter(values[:, 0], values[:, 1], s=22, color="teal")
    ax.set(title=title, xlabel="Feature 1", ylabel="Feature 2")
    # Equal aspect makes equal numerical distances look equal on both axes.
    ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()
print("Prepared shape:", X.shape)
print("Scaled means:", X.mean(axis=0).round(4))
print("Scaled standard deviations:", X.std(axis=0).round(4))

### Your explanation
Why was scaling useful here? Describe how scaling changes the relative contribution of the features to distance. Does the absence of missing values remove the need to inspect the data?

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 2 · Build an agglomerative hierarchy
Start with individual observations and merge groups according to a linkage rule. Choosing two groups here is an experiment, not information supplied as known class labels.

In [ ]:
# Average linkage uses the mean of all cross-group Euclidean distances.
agglomerative = AgglomerativeClustering(n_clusters=2, linkage="average")
agg_labels = agglomerative.fit_predict(X)
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=agg_labels, cmap="tab10", s=25)
plt.title("Agglomerative clustering · average linkage · 2 groups")
plt.xlabel("Standardised feature 1")
plt.ylabel("Standardised feature 2")
plt.axis("equal")
plt.tight_layout()
plt.show()
print("Group sizes:")
print(pd.Series(agg_labels).value_counts().sort_index())

# Optional contrast: bisecting K-means is one top-down splitting approach.
# It is not the DIANA algorithm and need not reverse agglomerative merges.
divisive = BisectingKMeans(n_clusters=2, bisecting_strategy="biggest_inertia", random_state=31)
div_labels = divisive.fit_predict(X)
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=div_labels, cmap="tab10", s=25)
plt.title("Top-down contrast · bisecting K-means · 2 groups")
plt.xlabel("Standardised feature 1")
plt.ylabel("Standardised feature 2")
plt.axis("equal")
plt.tight_layout()
plt.show()

### Your explanation
Explain bottom-up merging and top-down splitting. What choices do these two implementations make? Why might their partitions differ?

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 3 · Compare linkage methods
Keep X and the requested number of groups fixed. Single uses the closest pair, complete the furthest pair, average all cross-pairs, and Ward the increase in within-group SSE. Ward requires Euclidean geometry.

In [ ]:
# Change linkage only, so differences can be attributed to the linkage choice.
methods = ["single", "complete", "average", "ward"]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, method in zip(axes.flat, methods):
    labels = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=22)
    ax.set(title=f"{method.title()} linkage", xlabel="Standardised feature 1", ylabel="Standardised feature 2")
    ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

### Your explanation
Which linkage follows the curved structure most closely? Look for chaining and the effect of scattered observations. Explain why no linkage is universally best.

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 4 · Read a dendrogram
Use a small, separate dataset so leaf labels remain readable. The merge height follows the selected linkage. Horizontal leaf spacing has no distance meaning. We use average linkage here; Ward heights in SciPy use √(2 × SSE increase).

In [ ]:
# Small clouds make a readable dendrogram; we still do not use generator labels.
small_raw, _ = make_blobs(n_samples=15, centers=[[-3, -2], [0, 3], [3, -1]],
                          cluster_std=0.55, random_state=16)
small = StandardScaler().fit_transform(small_raw)
leaf_names = [chr(65 + i) for i in range(len(small))]
Z = linkage(small, method="average", metric="euclidean")
# This is an initial candidate. Change it and rerun to inspect another cut.
cut_height = 1.0
cut_labels = fcluster(Z, t=cut_height, criterion="distance")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dendrogram(Z, labels=leaf_names, color_threshold=cut_height, ax=axes[0])
axes[0].axhline(cut_height, color="crimson", linestyle="--", label=f"Cut = {cut_height}")
axes[0].set(title="Average-linkage dendrogram", xlabel="Observation (arbitrary leaf order)", ylabel="Linkage height")
axes[0].legend()
axes[1].scatter(small[:, 0], small[:, 1], c=cut_labels, cmap="tab10", s=45)
for name, point in zip(leaf_names, small):
    axes[1].annotate(name, point, xytext=(5, 5), textcoords="offset points")
axes[1].set(title="Groups below the cut", xlabel="Standardised feature 1", ylabel="Standardised feature 2")
axes[1].set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()
print("Groups at this cut:", len(np.unique(cut_labels)))
print(pd.DataFrame({"observation": leaf_names, "group": cut_labels}))

### Your explanation
Choose a second cut height and rerun. How does the group count change? Use the joining heights to justify a cut, and explain why leaf order is not an extra distance axis.

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 5 · Identify core, border and noise observations
DBSCAN uses ε and MinPts instead of a supplied group count. In scikit-learn, `min_samples` includes the observation itself. A border point can join a group but does not expand it. Noise is an outcome under the selected parameters, not proof of a data error.

In [ ]:
eps = 0.30
min_pts = 5
model = DBSCAN(eps=eps, min_samples=min_pts).fit(X)
db_labels = model.labels_
# scikit-learn uses -1 for noise; core_sample_indices_ identifies core points.
core_mask = np.zeros(len(X), dtype=bool)
core_mask[model.core_sample_indices_] = True
noise_mask = db_labels == -1
border_mask = (~core_mask) & (~noise_mask)

def plot_dbscan(ax, values, fitted, title):
    labels = fitted.labels_
    core = np.zeros(len(values), dtype=bool)
    core[fitted.core_sample_indices_] = True
    noise = labels == -1
    border = (~core) & (~noise)
    # Match colours to group IDs while using different shapes for point types.
    normalizer = plt.Normalize(vmin=0, vmax=max(1, labels.max()))
    for mask, marker, name, size in [(core, "o", "Core", 35), (border, "D", "Border", 25)]:
        if mask.any():
            ax.scatter(values[mask, 0], values[mask, 1], c=labels[mask], cmap="tab10",
                       norm=normalizer, marker=marker, s=size, edgecolor="black", linewidth=0.4, label=name)
    if noise.any():
        ax.scatter(values[noise, 0], values[noise, 1], c="grey", marker="x", s=35, label="Noise")
    ax.set(title=title, xlabel="Standardised feature 1", ylabel="Standardised feature 2")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(fontsize=8)

fig, ax = plt.subplots(figsize=(7, 5))
plot_dbscan(ax, X, model, f"DBSCAN · eps={eps} · MinPts={min_pts}")
plt.tight_layout()
plt.show()
counts = NearestNeighbors(radius=eps).fit(X).radius_neighbors(X, return_distance=False)
types = np.where(core_mask, "core", np.where(noise_mask, "noise", "border"))
report = pd.DataFrame({"observation": np.arange(1, len(X) + 1), "neighbours_including_self": [len(c) for c in counts],
                       "group": db_labels, "type": types})
print("Group count (excluding noise):", len(set(db_labels) - {-1}))
print("Core / border / noise counts:")
print(report["type"].value_counts())
print("\nExamples of each available point type:")
print(report.groupby("type").head(3).to_string(index=False))

### Your explanation
Choose examples from the report and explain their classifications using neighbour counts and core connectivity. Can a point previously treated as noise later become border during expansion?

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 6 · Investigate parameters and compare methods
Use neighbour distances as evidence for a candidate ε, then inspect several fits. The curve includes the observation itself in the neighbour count: MinPts = 5 uses the fifth distance including self. This avoids an off-by-one mismatch with DBSCAN.

In [ ]:
# Passing X explicitly to kneighbors includes the point itself at distance zero.
distances, _ = NearestNeighbors(n_neighbors=min_pts).fit(X).kneighbors(X)
k_distances = np.sort(distances[:, min_pts - 1])
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(X) + 1), k_distances, color="teal")
plt.axhline(eps, color="crimson", linestyle="--", label=f"Candidate eps={eps}")
plt.title(f"Distance to neighbour {min_pts} (including self)")
plt.xlabel("Observations sorted by neighbour distance")
plt.ylabel("Euclidean distance on prepared data")
plt.legend()
plt.tight_layout()
plt.show()

# Compare three radii and two neighbourhood-count requirements.
rows = []
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, mp in enumerate([4, 8]):
    for col, radius in enumerate([0.18, 0.30, 0.48]):
        fitted = DBSCAN(eps=radius, min_samples=mp).fit(X)
        labels = fitted.labels_
        rows.append({"eps": radius, "MinPts": mp, "groups": len(set(labels) - {-1}),
                     "noise": int(np.sum(labels == -1))})
        plot_dbscan(axes[row, col], X, fitted, f"eps={radius} · MinPts={mp}")
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).to_string(index=False))

# Compare methods using exactly the same prepared observations.
comparisons = {
    "K-means · k=2": KMeans(n_clusters=2, n_init=20, random_state=31).fit_predict(X),
    "Average agglomerative · k=2": AgglomerativeClustering(n_clusters=2, linkage="average").fit_predict(X)
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, labels) in zip(axes[:2], comparisons.items()):
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=22)
    ax.set(title=title, xlabel="Standardised feature 1", ylabel="Standardised feature 2")
    ax.set_aspect("equal", adjustable="datalim")
plot_dbscan(axes[2], X, DBSCAN(eps=eps, min_samples=min_pts).fit(X), "DBSCAN · chosen parameters")
plt.tight_layout()
plt.show()

### Your explanation
Justify a candidate ε and MinPts using the curve, plots and counts. Explain which method suits your intended grouping and why. How could one global ε struggle with regions of different densities? Why should we not simply treat noise as another ordinary cluster when scoring?

In [ ]:
# Write your explanation here, using the results above as evidence.
# Explain why you made the choice, not only what the code does.
# Your answer:

## Task 7 · Your turn: independent practice
Create the new synthetic dataset below, then carry out your own analysis. No worked solution is supplied.

1. Inspect observations, scales and missing values; justify any preprocessing.
2. Compare at least two linkage methods and interpret a dendrogram (use a small, explicitly selected subset if needed).
3. Investigate DBSCAN with at least three ε values and two MinPts settings. Show core, border and noise points.
4. Compare your chosen density-based result with K-means or agglomerative clustering on the same prepared data.
5. Explain which method you would use, what the groups could mean, and what remains uncertain. Investigate rather than automatically delete unusual points.
6. Write explanations within the notebook and include plots as evidence. Save the notebook and export the PDF for submission on your course platform.

The dataset has dense and sparse regions, scattered observations and features on different numerical scales. The starter supplies data only; you choose the preprocessing, methods and parameters.

In [ ]:
# Independent dataset: keep this separate from X used in guided activities.
practice_rng = np.random.default_rng(83)
dense_region = practice_rng.normal(loc=[-2.5, 0.0], scale=[0.20, 0.30], size=(65, 2))
sparse_region = practice_rng.normal(loc=[2.0, 0.6], scale=[0.85, 1.10], size=(65, 2))
scattered = practice_rng.uniform(low=[-4.0, -2.5], high=[4.0, 3.0], size=(12, 2))
practice_values = np.vstack([dense_region, sparse_region, scattered])
practice_values[:, 1] *= 25.0
practice_df = pd.DataFrame(practice_values, columns=["measurement_a", "measurement_b"])
# A row ID identifies an observation; it is not a clustering feature or group label.
practice_df.index = pd.Index(np.arange(1, len(practice_df) + 1), name="observation_id")
print("Independent dataset (first ten observations)")
print(practice_df.head(10))
print("Dataset shape:", practice_df.shape)
# Start your inspection and analysis in the empty cells below.

### Your work · Inspection and preprocessing

In [ ]:
# Add your code, plots and reasoning here.
# Justify your decisions using the independent dataset.

### Your work · Linkage and dendrogram investigation

In [ ]:
# Add your code, plots and reasoning here.
# Justify your decisions using the independent dataset.

### Your work · DBSCAN parameter investigation

In [ ]:
# Add your code, plots and reasoning here.
# Justify your decisions using the independent dataset.

### Your work · Comparison and justified conclusion

In [ ]:
# Add your code, plots and reasoning here.
# Justify your decisions using the independent dataset.